# Cette partie permet de lancer et utiliser Ollama avec Kaggle

In [ ]:
import os
os.chdir('/kaggle/working/')

#if not os.path.exists('/kaggle/tmp'):
#    os.mkdir('/kaggle/tmp')
#os.chdir('/kaggle/tmp/')

print(os.getcwd())

import subprocess
import os

def run(commands):
    for command in commands:
        with subprocess.Popen(command, shell = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, bufsize = 1) as sp:
            for line in sp.stdout:
                line = line.decode("utf-8", errors = "replace")
                if "undefined reference" in line:
                    raise RuntimeError("Failed Processing.")
                print(line, flush = True, end = "")
        pass
    pass
pass



In [ ]:
!pip install ollama

In [ ]:
!ollama

In [ ]:
commands = [
        "ollama pull mistral:7b",
        "ollama pull llama2:7b",
        "ollama pull deepseek-r1",
        ]
run(commands)

In [ ]:
commands = [
        "curl -fsSL https://ollama.com/install.sh | sh",
]
run(commands)

import os
os.system("/usr/local/bin/ollama serve &")
os.system("echo 'ollama test'")

# GSM8K

In [ ]:
import pandas as pd
import numpy as np
import ollama
import time
import warnings
import re
import random
from tqdm import tqdm # Import tqdm

warnings.filterwarnings("ignore")

# --- Configuration ---
MODEL_NAME = "llama2:7b" 
NUM_SHOTS = 8          # Number of few-shot examples
NUM_QUESTIONS_TO_PROCESS = 500
OUTPUT_FILENAME = f"{MODEL_NAME.replace(':','-')}_gsm8k.csv"
RANDOM_STATE = 40
# ---------------------


print("Loading training data for few-shot examples...")
splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])
shots = df_train.sample(NUM_SHOTS, random_state=RANDOM_STATE).to_dict(orient="records")

# Prepare the few-shot examples
shots_list = map(lambda x : "{question : " + x['question'] + "\nanswer : " + x['answer'] + "}", list(shots))
shots_str = str([shot for shot in list(shots_list)])
print(f"Prepared {NUM_SHOTS} few-shot examples.")

# Load the test dataset
print("Loading test data...")
df_test = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["test"])
questions = df_test["question"].tolist()
# Ensure we don't try to process more questions than available
num_to_process = min(NUM_QUESTIONS_TO_PROCESS, len(questions))
questions_to_process = questions[:num_to_process]
print(f"Loaded {len(questions)} test questions. Processing the first {num_to_process}.")


# --- System & User Prompt Setup ---
system_content = (
    "You are a math solver with extremely strict output formatting requirements.\n"
    "Your final answer must be one number, it must be at the end of your answer and preceded by '\\n####'.\n"
    "You are asked to follow the same format as the following examples:\n" + shots_str
)

# --- Main Processing Loop with tqdm ---
responses = []
print(f"Starting processing with model: {MODEL_NAME}")

# Wrap the questions list with tqdm for the progress bar
for question in tqdm(questions_to_process, desc="Processing Questions"):
    # User prompt containing only the new question
    user_prompt = f"question : {question}\nanswer :"

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_content},
                {'role': 'user', 'content': user_prompt}
            ]
        )
        answer = response["message"]["content"].strip()

    except Exception as e:
        # Log error in the response list and print a message (tqdm handles progress bar)
        print(f"\nError processing question: '{question[:50]}...'. Error: {e}")
        answer = f"Error: {e}"

    responses.append(answer)

# --- Post-processing ---
print("\nProcessing complete. Extracting answers...")

def extract_final_answer(text):
    # Searches for #### followed by optional whitespace, then captures digits (allowing for internal commas)
    match = re.search(r'####\s*([\d,]+)', text)
    if match:
        return match.group(1).replace(',', '') # Remove commas
    # Fallback: try to find the last number in the string if the specific format fails
    numbers = re.findall(r'\d+', text)
    if numbers:
      # Try to be a bit smarter: avoid year-like numbers if others are present, unless it's the only one
      if len(numbers) > 1:
          non_year_numbers = [n for n in numbers if not (len(n) == 4 and n.startswith(('19', '20')))]
          if non_year_numbers:
              return non_year_numbers[-1]
      return numbers[-1] # Return the last number found
    return None # Return None if no number is found

# Extract the true answers from the test set slice used
true_answers_raw = df_test['answer'][:num_to_process].tolist()
true_answers_extracted = [extract_final_answer(ans) for ans in true_answers_raw]

# Build the DataFrame
result_df = pd.DataFrame({
    'question': questions_to_process,
    'response_raw': responses,
    'response_extracted': [extract_final_answer(resp) for resp in responses],
    'true_answer_extracted': true_answers_extracted
})

# Save to CSV
result_df.to_csv(OUTPUT_FILENAME, index=False)
print(f"✅ Results saved to {OUTPUT_FILENAME}")

Loading training data for few-shot examples...
Prepared 8 few-shot examples.
Loading test data...
Loaded 1319 test questions. Processing the first 500.
Starting processing with model: llama2:7b
Processing Questions: 100%|██████████| 500/500 [25:30<00:00,  3.06s/it]

Processing complete. Exracting answers...
✅ Results saved to llama2-7b_gsm8k.csv


In [ ]:
import pandas as pd
import numpy as np
import ollama
import time
import warnings
import re
import random
from tqdm import tqdm # Import tqdm

warnings.filterwarnings("ignore")

# --- Configuration ---
MODEL_NAME = "mistral:7b" 
NUM_SHOTS = 8          # Number of few-shot examples
NUM_QUESTIONS_TO_PROCESS = 500
OUTPUT_FILENAME = f"{MODEL_NAME.replace(':','-')}_gsm8k.csv"
RANDOM_STATE = 40
# ---------------------


print("Loading training data for few-shot examples...")
splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])
shots = df_train.sample(NUM_SHOTS, random_state=RANDOM_STATE).to_dict(orient="records")

# Prepare the few-shot examples
shots_list = map(lambda x : "{question : " + x['question'] + "\nanswer : " + x['answer'] + "}", list(shots))
shots_str = str([shot for shot in list(shots_list)])
print(f"Prepared {NUM_SHOTS} few-shot examples.")

# Load the test dataset
print("Loading test data...")
df_test = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["test"])
questions = df_test["question"].tolist()
# Ensure we don't try to process more questions than available
num_to_process = min(NUM_QUESTIONS_TO_PROCESS, len(questions))
questions_to_process = questions[:num_to_process]
print(f"Loaded {len(questions)} test questions. Processing the first {num_to_process}.")


# --- System & User Prompt Setup ---
system_content = (
    "You are a math solver with extremely strict output formatting requirements.\n"
    "Your final answer must be one number, it must be at the end of your answer and preceded by '\\n####'.\n"
    "You are asked to follow the same format as the following examples:\n" + shots_str
)

# --- Main Processing Loop with tqdm ---
responses = []
print(f"Starting processing with model: {MODEL_NAME}")

# Wrap the questions list with tqdm for the progress bar
for question in tqdm(questions_to_process, desc="Processing Questions"):
    # User prompt containing only the new question
    user_prompt = f"question : {question}\nanswer :"

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_content},
                {'role': 'user', 'content': user_prompt}
            ]
        )
        answer = response["message"]["content"].strip()

    except Exception as e:
        # Log error in the response list and print a message (tqdm handles progress bar)
        print(f"\nError processing question: '{question[:50]}...'. Error: {e}")
        answer = f"Error: {e}"

    responses.append(answer)

# --- Post-processing ---
print("\nProcessing complete. Extracting answers...")

def extract_final_answer(text):
    # Searches for #### followed by optional whitespace, then captures digits (allowing for internal commas)
    match = re.search(r'####\s*([\d,]+)', text)
    if match:
        return match.group(1).replace(',', '') # Remove commas
    # Fallback: try to find the last number in the string if the specific format fails
    numbers = re.findall(r'\d+', text)
    if numbers:
      # Try to be a bit smarter: avoid year-like numbers if others are present, unless it's the only one
      if len(numbers) > 1:
          non_year_numbers = [n for n in numbers if not (len(n) == 4 and n.startswith(('19', '20')))]
          if non_year_numbers:
              return non_year_numbers[-1]
      return numbers[-1] # Return the last number found
    return None # Return None if no number is found

# Extract the true answers from the test set slice used
true_answers_raw = df_test['answer'][:num_to_process].tolist()
true_answers_extracted = [extract_final_answer(ans) for ans in true_answers_raw]

# Build the DataFrame
result_df = pd.DataFrame({
    'question': questions_to_process,
    'response_raw': responses,
    'response_extracted': [extract_final_answer(resp) for resp in responses],
    'true_answer_extracted': true_answers_extracted
})

# Save to CSV
result_df.to_csv(OUTPUT_FILENAME, index=False)
print(f"✅ Results saved to {OUTPUT_FILENAME}")

Loading training data for few-shot examples...
Prepared 8 few-shot examples.
Loading test data...
Loaded 1319 test questions. Processing the first 500.
Starting processing with model: mistral:7b
Processing Questions: 100%|██████████| 500/500 [31:05<00:00,  3.73s/it]

Processing complete. Exracting answers...
✅ Results saved to mistral-7b_gsm8k.csv


In [ ]:
import pandas as pd
import numpy as np
import ollama
import time
import warnings
import re
import random
from tqdm import tqdm # Import tqdm

warnings.filterwarnings("ignore")

# --- Configuration ---
MODEL_NAME = "deepseek-r1" 
NUM_SHOTS = 8          # Number of few-shot examples
NUM_QUESTIONS_TO_PROCESS = 500
OUTPUT_FILENAME = f"{MODEL_NAME.replace(':','-')}_gsm8k.csv"
RANDOM_STATE = 40
# ---------------------


print("Loading training data for few-shot examples...")
splits = {'train': 'main/train-00000-of-00001.parquet', 'test': 'main/test-00000-of-00001.parquet'}
df_train = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["train"])
shots = df_train.sample(NUM_SHOTS, random_state=RANDOM_STATE).to_dict(orient="records")

# Prepare the few-shot examples
shots_list = map(lambda x : "{question : " + x['question'] + "\nanswer : " + x['answer'] + "}", list(shots))
shots_str = str([shot for shot in list(shots_list)])
print(f"Prepared {NUM_SHOTS} few-shot examples.")

# Load the test dataset
print("Loading test data...")
df_test = pd.read_parquet("hf://datasets/openai/gsm8k/" + splits["test"])
questions = df_test["question"].tolist()
# Ensure we don't try to process more questions than available
num_to_process = min(NUM_QUESTIONS_TO_PROCESS, len(questions))
questions_to_process = questions[:num_to_process]
print(f"Loaded {len(questions)} test questions. Processing the first {num_to_process}.")


# --- System & User Prompt Setup ---
system_content = (
    "You are a math solver with extremely strict output formatting requirements.\n"
    "Your final answer must be one number, it must be at the end of your answer and preceded by '\\n####'.\n"
    "You are asked to follow the same format as the following examples:\n" + shots_str
)

# --- Main Processing Loop with tqdm ---
responses = []
print(f"Starting processing with model: {MODEL_NAME}")

# Wrap the questions list with tqdm for the progress bar
for question in tqdm(questions_to_process, desc="Processing Questions"):
    # User prompt containing only the new question
    user_prompt = f"question : {question}\nanswer :"

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_content},
                {'role': 'user', 'content': user_prompt}
            ]
        )
        answer = response["message"]["content"].strip()

    except Exception as e:
        # Log error in the response list and print a message (tqdm handles progress bar)
        print(f"\nError processing question: '{question[:50]}...'. Error: {e}")
        answer = f"Error: {e}"

    responses.append(answer)

# --- Post-processing ---
print("\nProcessing complete. Extracting answers...")

def extract_final_answer(text):
    # Searches for #### followed by optional whitespace, then captures digits (allowing for internal commas)
    match = re.search(r'####\s*([\d,]+)', text)
    if match:
        return match.group(1).replace(',', '') # Remove commas
    # Fallback: try to find the last number in the string if the specific format fails
    numbers = re.findall(r'\d+', text)
    if numbers:
      # Try to be a bit smarter: avoid year-like numbers if others are present, unless it's the only one
      if len(numbers) > 1:
          non_year_numbers = [n for n in numbers if not (len(n) == 4 and n.startswith(('19', '20')))]
          if non_year_numbers:
              return non_year_numbers[-1]
      return numbers[-1] # Return the last number found
    return None # Return None if no number is found

# Extract the true answers from the test set slice used
true_answers_raw = df_test['answer'][:num_to_process].tolist()
true_answers_extracted = [extract_final_answer(ans) for ans in true_answers_raw]

# Build the DataFrame
result_df = pd.DataFrame({
    'question': questions_to_process,
    'response_raw': responses,
    'response_extracted': [extract_final_answer(resp) for resp in responses],
    'true_answer_extracted': true_answers_extracted
})

# Save to CSV
result_df.to_csv(OUTPUT_FILENAME, index=False)
print(f"✅ Results saved to {OUTPUT_FILENAME}")

Loading training data for few-shot examples...
Prepared 8 few-shot examples.
Loading test data...
Loaded 1319 test questions. Processing the first 500.
Starting processing with model: deepseek-r1
Processing Questions: 100%|██████████| 500/500 [2:12:35<00:00,  15.91s/it]

Processing complete. Exracting answers...
✅ Results saved to deepseek-r1_gsm8k.csv
